In [ ]:
import os
import sys
sys.path.append("/media/ps/ssd5/ayr/x-nemo-inference")
from decord import VideoReader
import mediapipe as mp
import numpy as np
import cv2
import torch
from scripts.vid2pose import extract_bbox_mp
from scipy.ndimage import gaussian_filter1d


def scale_bb(bbox, scale, size):
    left, top, right, bot = bbox[:4].tolist()
    width = right - left
    height = bot - top
    length = max(width, height) * scale
    center_X = (left + right) * 0.5
    center_Y = (top + bot) * 0.5
    left, top, right, bot = [
        center_X - length / 2,
        center_Y - length / 2,
        center_X + length / 2,
        center_Y + length / 2,
    ]
    if check_oob((left, top, right, bot), size):
        return bbox
    else:
        return np.array([left, top, right, bot])
    

def get_bbox_param(bbox, ref_bbox):
    left, top, right, bot = bbox
    center = np.array([(bot + top) * 0.5, (left + right) * 0.5])
    length = max(right - left, bot - top)

    ref_left, ref_top, ref_right, ref_bot = ref_bbox
    ref_center = np.array([(ref_bot + ref_top) * 0.5, (ref_left + ref_right) * 0.5])
    ref_length = max(ref_right - ref_left, ref_bot - ref_top)  # 最好保证bbox是正方形

    return np.asarray(
        ((center - ref_center) / ref_length).tolist() + [length / ref_length]
    )


def check_oob(bbox, size):
    left, top, right, bot = bbox
    return left < 0 or top < 0 or right > size[1] - 1 or bot > size[0] - 1


pose_vid_path = "/media/ps/ssd5/ayr/x-nemo-inference/demo/driving_video.mp4"
control = VideoReader(pose_vid_path)
fps = control.get_avg_fps() # 25
control = control.get_batch(range(len(control))).asnumpy()
print("hello")


options = mp.tasks.vision.FaceDetectorOptions(
                base_options=mp.tasks.BaseOptions(
                    model_asset_path='/media/ps/ssd5/ayr/x-nemo-inference/blaze_face_short_range.tflite'
                ),
                running_mode=mp.tasks.vision.RunningMode.VIDEO,
            )
detector = mp.tasks.vision.FaceDetector.create_from_options(options)
timestamp_ms = 0
ani_bbox_center = []
fix_length = None
bbox_params = []
timestamp_ms = 0
print(control.shape)

for idx_control in range(control.shape[0]):
    pose_image = control[idx_control][:, : control[0].shape[0]]

    timestamp_ms += int(1000 / fps)
    bbox = extract_bbox_mp(pose_image, None, detector, timestamp_ms)

    bbox_param = torch.from_numpy(get_bbox_param(bbox, ref_bbox))
    bbox_param = torch.ones_like(bbox_param).to(dtype=bbox_param.dtype)  #########
    bbox_param[:2] *= 0  #########
    bbox_params.append(bbox_param)

    if idx_control == 0:
        bbox = scale_bb(bbox, scale=1.1, size=pose_image.shape[:2])
        fix_length = (
            round(bbox[2] - bbox[0]) // 2 * 2,
            round(bbox[3] - bbox[1]) // 2 * 2,
        )
    left, top, right, bot = bbox
    center_X = (left + right) * 0.5
    center_Y = (top + bot) * 0.5
    ani_bbox_center.append([center_X, center_Y])

ani_bbox_center = np.asarray(ani_bbox_center, dtype=np.float32)
ani_bbox_center = gaussian_filter1d(
    ani_bbox_center, sigma=5.0, axis=0
).tolist()
bbox_params = torch.stack(bbox_params, dim=0)

In [5]:
import torch
bbox_param = torch.ones(4, 3)
bbox_param[:,:2] *= 0
print(bbox_param)

tensor([[0., 0., 1.],
        [0., 0., 1.],
        [0., 0., 1.],
        [0., 0., 1.]])
